In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import warnings, time
warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
sns.set_theme(style="whitegrid", context="talk")
print("Imports loaded.")


In [ ]:
df = pd.read_csv('hand_gesture_clean_labeled.csv')
print(f"Shape: {df.shape}")
drop_cols = [c for c in ['person_name', 'gesture_label.1', 'gesture', 'output_label'] if c in df.columns]
df.drop(columns=drop_cols, inplace=True)
print(f"Dropped: {drop_cols}")
dist_cols = [c for c in df.columns if c.startswith("dist_")]
X = df[dist_cols].values
print(f"Samples: {len(df)}, Features: {len(dist_cols)}")


In [ ]:
df['joint_label'] = df['gesture_label'].astype(str) + '_' + df['person_id'].astype(str)
le_joint = LabelEncoder()
df['joint_encoded'] = le_joint.fit_transform(df['joint_label'])
y = df['joint_encoded'].values
n_classes = len(np.unique(y))
print(f"Unique joint classes: {n_classes}")
print(f"Target range: {y.min()}-{y.max()}")
vc = df['joint_label'].value_counts()
print(f"Min samples/class: {vc.min()}, Max: {vc.max()}, Classes <=1: {(vc<=1).sum()}")


In [ ]:
def get_model(name):
    if name == "Naive Bayes": return GaussianNB()
    elif name == "Decision Tree": return DecisionTreeClassifier(random_state=42)
    elif name == "XGBoost": return XGBClassifier(n_estimators=30, learning_rate=0.1, max_depth=6,
                             eval_metric='mlogloss', random_state=42, verbosity=0)
    elif name == "Random Forest": return RandomForestClassifier(n_estimators=80, n_jobs=-1, random_state=42)
    elif name == "KNN": return KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    elif name == "Logistic Regression": return LogisticRegression(max_iter=2000, random_state=42, n_jobs=-1)
    elif name == "SVM": return LinearSVC(max_iter=2000, random_state=42, dual=False)

model_names = ["Naive Bayes", "Decision Tree", "XGBoost", "Random Forest",
               "KNN", "Logistic Regression", "SVM"]
model_order = list(model_names)


In [ ]:
def scale_and_predict(model, X_tr, X_te, y_train, y_test, name):
    if name in ["Logistic Regression", "SVM", "KNN"]:
        X_tr_s = std_scaler.transform(X_tr)
        X_te_s = std_scaler.transform(X_te)
    elif name == "Naive Bayes":
        X_tr_s = mm_scaler.transform(X_tr)
        X_te_s = mm_scaler.transform(X_te)
    else:
        X_tr_s, X_te_s = X_tr, X_te
    model.fit(X_tr_s, y_train)
    preds = model.predict(X_te_s)
    return [accuracy_score(y_test, preds),
            precision_score(y_test, preds, average="weighted", zero_division=0),
            recall_score(y_test, preds, average="weighted", zero_division=0),
            f1_score(y_test, preds, average="weighted", zero_division=0)]


## 3.1 Variation of Training Ratio (Fixed Data Size = 100%)

In [ ]:
ratios = [0.5, 0.6, 0.7, 0.8]
results_3_1 = []
t0 = time.time()
for ratio in ratios:
    tp = int(ratio * 100)
    print(f"\n--- Split {tp}-{100-tp} ---")
    sss = StratifiedShuffleSplit(n_splits=2, train_size=ratio, random_state=42)
    for fold, (tr_idx, te_idx) in enumerate(sss.split(X, y), 1):
        ft = time.time()
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]
        std_scaler = StandardScaler().fit(X_tr)
        mm_scaler = MinMaxScaler().fit(X_tr)
        for name in model_names:
            try:
                model = get_model(name)
                m = scale_and_predict(model, X_tr, X_te, y_tr, y_te, name)
                results_3_1.append([f"{tp}-{100-tp}", name] + m)
            except Exception as e:
                print(f"  {name} err: {e}")
                results_3_1.append([f"{tp}-{100-tp}", name, 0.0, 0.0, 0.0, 0.0])
        print(f"  Fold {fold}: {time.time()-ft:.0f}s")
df_3_1 = pd.DataFrame(results_3_1, columns=["Split","Model","Accuracy","Precision","Recall","F1"])
dt = time.time()-t0
print(f"\nDone in {dt:.0f}s ({dt/60:.1f} min)")
df_3_1.to_csv("Experiment3_Joint_TrainingRatio.csv", index=False)
final_3_1 = df_3_1.groupby(["Split","Model"], as_index=False).mean().round(4)
print(final_3_1.to_string())


## 3.2 Variation of Data Size (Fixed Training Ratio = 80-20)

In [ ]:
data_sizes = [0.4, 0.6, 0.8, 1.0]
results_3_2 = []
t0 = time.time()
for size in data_sizes:
    sp = int(size * 100)
    print(f"\n--- Data Size {sp}% ---")
    df_s = df.groupby("joint_encoded", group_keys=False).apply(lambda x: x.sample(frac=size, random_state=42))
    X_s, y_s = df_s[dist_cols].values, df_s['joint_encoded'].values
    sss = StratifiedShuffleSplit(n_splits=2, train_size=0.8, random_state=42)
    for fold, (tr_idx, te_idx) in enumerate(sss.split(X_s, y_s), 1):
        ft = time.time()
        X_tr, X_te = X_s[tr_idx], X_s[te_idx]
        y_tr, y_te = y_s[tr_idx], y_s[te_idx]
        std_scaler = StandardScaler().fit(X_tr)
        mm_scaler = MinMaxScaler().fit(X_tr)
        for name in model_names:
            try:
                model = get_model(name)
                m = scale_and_predict(model, X_tr, X_te, y_tr, y_te, name)
                results_3_2.append([f"{sp}%", name] + m)
            except Exception as e:
                print(f"  {name} err: {e}")
                results_3_2.append([f"{sp}%", name, 0.0, 0.0, 0.0, 0.0])
        print(f"  Fold {fold}: {time.time()-ft:.0f}s")
df_3_2 = pd.DataFrame(results_3_2, columns=["Data_Size","Model","Accuracy","Precision","Recall","F1"])
dt = time.time()-t0
print(f"\nDone in {dt:.0f}s ({dt/60:.1f} min)")
df_3_2.to_csv("Experiment3_Joint_DataSize.csv", index=False)
final_3_2 = df_3_2.groupby(["Data_Size","Model"], as_index=False).mean().round(4)
print(final_3_2.to_string())


In [ ]:
def bar_plot(df, x_col, title, xlabel, fn, ylim=(0,1.05)):
    fig, axs = plt.subplots(2,2,figsize=(16,10))
    for i,met in enumerate(["Accuracy","Precision","Recall","F1"]):
        sns.barplot(data=df, x=x_col, y=met, hue="Model", hue_order=model_order,
                    errorbar=None, palette="tab10", ax=axs[i//2,i%2])
        axs[i//2,i%2].set_title(f"{met} vs {title}", fontweight="bold")
        axs[i//2,i%2].set_ylim(*ylim); axs[i//2,i%2].legend_.remove(); axs[i//2,i%2].set_xlabel(xlabel)
    h,_ = axs[0,0].get_legend_handles_labels()
    fig.legend(h, model_order, title="Model", loc="center left", bbox_to_anchor=(1.02,0.5))
    plt.suptitle(f"Joint ID - {title}", fontsize=16, fontweight="bold")
    plt.tight_layout(rect=[0,0,1,1]); plt.savefig(fn, dpi=300, bbox_inches='tight'); plt.show()

def line_plot(df, x_col, title, xlabel, fn, ylim=(0,1.05)):
    fig, axs = plt.subplots(2,2,figsize=(14,10))
    for i,met in enumerate(["Accuracy","Precision","Recall","F1"]):
        piv = df.pivot_table(index=x_col, columns="Model", values=met, aggfunc="mean")
        piv.plot(marker="o", ax=axs[i//2,i%2])
        axs[i//2,i%2].set_title(f"{met} Trend", fontweight="bold")
        axs[i//2,i%2].set_ylim(*ylim); axs[i//2,i%2].legend_.remove()
        axs[i//2,i%2].set_xlabel(xlabel); axs[i//2,i%2].grid(True, alpha=0.3)
    h,_ = axs[0,0].get_legend_handles_labels()
    fig.legend(h, model_order, title="Model", loc="center left", bbox_to_anchor=(1.02,0.5))
    plt.suptitle(f"Joint ID - {title} Trend", fontsize=16, fontweight="bold")
    plt.tight_layout(rect=[0,0,1,1]); plt.savefig(fn, dpi=300, bbox_inches='tight'); plt.show()

def box_plot(df, x_col, title, fn, ylim=(0,1.05)):
    fig, axs = plt.subplots(2,2,figsize=(16,10))
    for i,met in enumerate(["Accuracy","Precision","Recall","F1"]):
        sns.boxplot(data=df, x=x_col, y=met, hue="Model", hue_order=model_order,
                    palette="tab10", ax=axs[i//2,i%2])
        axs[i//2,i%2].set_title(f"{met} Distribution", fontweight="bold")
        axs[i//2,i%2].set_ylim(*ylim); axs[i//2,i%2].legend_.remove()
    h,_ = axs[0,0].get_legend_handles_labels()
    fig.legend(h, model_order, title="Model", loc="center left", bbox_to_anchor=(1.02,0.5))
    plt.suptitle(f"Joint ID - {title} (Fold Variance)", fontsize=16, fontweight="bold")
    plt.tight_layout(rect=[0,0,1,1]); plt.savefig(fn, dpi=300, bbox_inches='tight'); plt.show()

def rank_plot(df, title, fn):
    best = df.groupby("Model")["Accuracy"].max().sort_values(ascending=False)
    fig,ax = plt.subplots(figsize=(10,5))
    colors = ["#2ecc71" if i==0 else "#95a5a6" for i in range(len(best))]
    bars = ax.barh(best.index, best.values, color=colors)
    for b,v in zip(bars,best.values): ax.text(v+0.005, b.get_y()+b.get_height()/2, f"{v:.4f}", va="center")
    ax.set_xlim(0,1.05); ax.set_xlabel("Best Accuracy")
    ax.set_title(f"Model Ranking - {title}", fontweight="bold")
    plt.tight_layout(); plt.savefig(fn, dpi=300, bbox_inches='tight'); plt.show()

def heat_plot(df, x_col, title, fn):
    fig, axs = plt.subplots(2,2,figsize=(14,10))
    for i,met in enumerate(["Accuracy","Precision","Recall","F1"]):
        piv = df.pivot_table(index="Model", columns=x_col, values=met, aggfunc="mean")
        sns.heatmap(piv, annot=True, fmt=".3f", cmap="YlGnBu", linewidths=0.5, ax=axs[i//2,i%2])
        axs[i//2,i%2].set_title(f"{met} Heatmap", fontweight="bold")
    plt.suptitle(f"Joint ID - {title}", fontsize=16, fontweight="bold")
    plt.tight_layout(rect=[0,0,1,0.95]); plt.savefig(fn, dpi=300, bbox_inches='tight'); plt.show()

def summary_stats(df, group_col, title):
    print("="*90); print(f"SUMMARY - {title}"); print("="*90)
    s = df.groupby(["Model",group_col]).agg(
        Mean_Acc=("Accuracy","mean"),Std_Acc=("Accuracy","std"),
        Mean_Prec=("Precision","mean"),Mean_Rec=("Recall","mean"),Mean_F1=("F1","mean")).round(4)
    print(s.to_string())
    best = df.groupby("Model")["Accuracy"].max().sort_values(ascending=False)
    print(f"\nBEST: {best.index[0]} ({best.iloc[0]:.4f}) | WORST: {best.index[-1]} ({best.iloc[-1]:.4f})\n")


## Visualizations: Training Ratio Variation

In [ ]:
m31 = df_3_1.groupby(["Split","Model"], as_index=False).mean()
bar_plot(m31, "Split", "Training Ratio", "Train-Test Split", "Exp3_1_Bar_TrainingRatio.png")
line_plot(df_3_1, "Split", "Training Ratio", "Train-Test Split", "Exp3_1_Line_TrainingRatio.png")
box_plot(df_3_1, "Split", "Training Ratio", "Exp3_1_Box_TrainingRatio.png")
heat_plot(df_3_1, "Split", "Training Ratio Heatmap", "Exp3_1_Heatmap_Joint.png")
rank_plot(df_3_1, "Training Ratio", "Exp3_1_Ranking_TrainingRatio.png")
summary_stats(df_3_1, "Split", "Exp 3.1 - Training Ratio Variation")


## Visualizations: Data Size Variation

In [ ]:
m32 = df_3_2.groupby(["Data_Size","Model"], as_index=False).mean()
bar_plot(m32, "Data_Size", "Data Size", "Data Size", "Exp3_2_Bar_DataSize.png")
line_plot(df_3_2, "Data_Size", "Data Size", "Data Size", "Exp3_2_Line_DataSize.png")
box_plot(df_3_2, "Data_Size", "Data Size", "Exp3_2_Box_DataSize.png")
heat_plot(df_3_2, "Data_Size", "Data Size Heatmap", "Exp3_2_Heatmap_Joint.png")
rank_plot(df_3_2, "Data Size", "Exp3_2_Ranking_DataSize.png")
summary_stats(df_3_2, "Data_Size", "Exp 3.2 - Data Size Variation")


## Confusion Matrix for Best Model

In [ ]:
ss = StratifiedShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
tr_idx, te_idx = next(ss.split(X, y))
X_tr, X_te = X[tr_idx], X[te_idx]
y_tr, y_te = y[tr_idx], y[te_idx]
std_scaler = StandardScaler().fit(X_tr)
mm_scaler = MinMaxScaler().fit(X_tr)

best_name = df_3_1.groupby("Model")["Accuracy"].mean().idxmax()
print(f"Best model: {best_name}")

model = get_model(best_name)
if best_name in ["Logistic Regression", "SVM", "KNN"]:
    X_tr_s = std_scaler.transform(X_tr); X_te_s = std_scaler.transform(X_te)
elif best_name == "Naive Bayes":
    X_tr_s = mm_scaler.transform(X_tr); X_te_s = mm_scaler.transform(X_te)
else:
    X_tr_s, X_te_s = X_tr, X_te

model.fit(X_tr_s, y_tr)
preds = model.predict(X_te_s)
acc = accuracy_score(y_te, preds)
print(f"Test accuracy: {acc:.4f}")

top_classes = np.unique(y_te)[:10]
mask_te = np.isin(y_te, top_classes)
y_te_sub = y_te[mask_te]
preds_sub = preds[mask_te]
cm = confusion_matrix(y_te_sub, preds_sub)
fig,ax = plt.subplots(figsize=(10,8))
disp = ConfusionMatrixDisplay(cm, display_labels=[le_joint.classes_[i] for i in top_classes])
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title(f"Confusion Matrix (10 classes) - {best_name}", fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.savefig("Exp3_Confusion_Matrix.png", dpi=300, bbox_inches='tight'); plt.show()
print("Done!")


## Cross-Experiment Comparison (Exp 1 vs Exp 2 vs Exp 3)

In [ ]:
dfs = []
try:
    d1 = pd.read_csv("Experiment1_HandActivity_TrainingRatio.csv")
    d1 = d1.groupby("Model")["Accuracy"].mean().reset_index()
    d1["Experiment"] = "Exp1: Gesture (5 cls)"
    dfs.append(d1)
except: pass
try:
    d2 = pd.read_csv("Experiment2_PersonID_TrainingRatio.csv")
    d2 = d2.groupby("Model")["Accuracy"].mean().reset_index()
    d2["Experiment"] = "Exp2: Person (102 cls)"
    dfs.append(d2)
except: pass
d3 = df_3_1.groupby("Model")["Accuracy"].mean().reset_index()
d3["Experiment"] = "Exp3: Joint (506 cls)"
dfs.append(d3)

comp = pd.concat(dfs, ignore_index=True)
fig,ax = plt.subplots(figsize=(14,6))
order = sorted(comp["Experiment"].unique())
sns.barplot(data=comp, x="Model", y="Accuracy", hue="Experiment", palette="Set2", ax=ax, hue_order=order)
ax.set_title("Cross-Experiment Accuracy Comparison", fontweight="bold")
ax.set_ylim(0,1.05); ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
for c in ax.containers: ax.bar_label(c, fmt="%.3f", fontsize=8)
plt.tight_layout(); plt.savefig("Exp3_Cross_Experiment_Comparison.png", dpi=300, bbox_inches='tight'); plt.show()
print("Done.")


## Overall Conclusion

In [ ]:
print("="*90); print("OVERALL BEST ACCURACY - JOINT IDENTIFICATION"); print("="*90)
b1 = df_3_1.groupby("Model")["Accuracy"].max().sort_values(ascending=False)
b2 = df_3_2.groupby("Model")["Accuracy"].max().sort_values(ascending=False)
print("\nBy Training Ratio:"); [(print(f"  {m:25s}: {a:.4f}"),) for m,a in b1.items()]
print("\nBy Data Size:"); [(print(f"  {m:25s}: {a:.4f}"),) for m,a in b2.items()]
all_best = pd.concat([b1,b2]).groupby(level=0).max().sort_values(ascending=False)
print(f"\nBest Model Overall: {all_best.index[0]} ({all_best.iloc[0]:.4f})")

fig,ax = plt.subplots(figsize=(12,5))
x = np.arange(len(all_best)); w = 0.35
ax.bar(x-w/2, [b1.get(m,0) for m in all_best.index], w, label="Training Ratio", color="#3498db")
ax.bar(x+w/2, [b2.get(m,0) for m in all_best.index], w, label="Data Size", color="#e74c3c")
ax.set_xticks(x); ax.set_xticklabels(all_best.index, rotation=30, ha="right")
ax.set_ylabel("Best Accuracy"); ax.set_title("Joint ID: Best Accuracy Comparison", fontweight="bold")
ax.legend(); ax.set_ylim(0,1.05)
plt.tight_layout(); plt.savefig("Exp3_Overall_Comparison.png", dpi=300, bbox_inches='tight'); plt.show()
print("\nAll plots and CSVs saved.")
